In [6]:
import sys
sys.path.append('..')
from sqlalchemy import create_engine, text
import os

from config import RUTA_UNIDAD_ONE_DRIVE
from config import RUTA_LOCAL_ONE_DRIVE
from config import API_AMIGOCLOUD_TOKEN_ADM
from config import POSTGRES_UTEA

POSTGRES_UTEA['DATABASE'] = 'utea_precision'

In [7]:
def obtener_engine():
    return create_engine(
        f"postgresql+psycopg2://{POSTGRES_UTEA['USER']}:{POSTGRES_UTEA['PASSWORD']}@{POSTGRES_UTEA['HOST']}:{POSTGRES_UTEA['PORT']}/{POSTGRES_UTEA['DATABASE']}"
    )

In [13]:
def insert_datos_lote_con_verificacion(lista_fechas):
    engine = obtener_engine()
    
    for fecha_shp in lista_fechas:
        # 1. Query de verificación
        query_check = text("""
            SELECT 1 FROM datos_cosecha.fechas
            WHERE fecha = :fech 
        """)
        
        # 2. Query de inserción
        query_insert = text("""
            INSERT INTO datos_cosecha.fechas 
            (fecha)
            VALUES (:fech)
        """)
        
        try:
            with engine.begin() as conn:
                # Ejecutar verificación
                existe = conn.execute(query_check, {'fech': fecha_shp}).fetchone()
                
                if not existe:
                    conn.execute(query_insert, {
                        'fech': fecha_shp
                    })
                    print(f"✅ Insertado: {fecha_shp}")
                else:
                    print(f"⚠️ Omitido (ya existe): {fecha_shp}")
                    
        except Exception as e:
            print(f"❌ Error en {fecha_shp}: {e}")

In [14]:
PATH_SHPS = RUTA_UNIDAD_ONE_DRIVE + r"\Ingenio Azucarero Guabira S.A\UTEA - SEMANAL - PROGRAMA DE COSECHA\2026\SEGUIMIENTO_COSECHA\SHP_RECORRIDOS"

In [15]:
# lista de nombres de archivos en ruta
fecha_shp = [
    os.path.splitext(archivo)[0] # separa nombre de archivo y extencion
    for archivo in os.listdir(PATH_SHPS) # recorre la lista
    if archivo.endswith('.shp') # solo los que terminan en .shp
]
# Ver el resultado
fecha_shp

['2026-04-23']

In [16]:
insert_datos_lote_con_verificacion(fecha_shp)

✅ Insertado: 2026-04-23
